In [1]:
# Cell 1) 패키지 / DB 설정
import urllib.parse
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "",
}

def get_engine(cfg=DB_CONFIG):
    user = cfg["user"]
    password = urllib.parse.quote_plus(cfg["password"])
    host = cfg["host"]
    port = cfg["port"]
    dbname = cfg["dbname"]
    conn_str = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}"
    return create_engine(conn_str, pool_pre_ping=True)

engine = get_engine()

SRC_SCHEMA = "d1_machine_log"
SRC_TABLES = [
    "fct1_machine_log",
    "fct2_machine_log",
    "fct3_machine_log",
    "fct4_machine_log",
]

TARGET_SCHEMA = "d1_machine_log"
TARGET_TABLE  = "fct_op_gap"

TEST_OK_PREFIX = "TEST RESULT :: OK"
TEST_NG_PREFIX = "TEST RESULT :: NG"
ON_TEXT = "제품 안착 완료 ON"

In [2]:
def find_fct_machine_log_tables(engine, schema: str) -> list[str]:
    sql = text("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = :schema
          AND table_type = 'BASE TABLE'
    """)
    df = pd.read_sql(sql, engine, params={"schema": schema})

    picked = []
    for t in df["table_name"]:
        # 정확히 FCT1~4_machine_log 만 허용
        if t in (
            "FCT1_machine_log",
            "FCT2_machine_log",
            "FCT3_machine_log",
            "FCT4_machine_log",
        ):
            picked.append(t)

    # FCT 번호 순서 정렬
    picked = sorted(picked, key=lambda x: int(x[3]))
    return picked


src_tables = find_fct_machine_log_tables(engine, SRC_SCHEMA)
print("Detected source tables:", src_tables)

if not src_tables:
    raise RuntimeError("FCT1~4_machine_log 테이블을 찾지 못했습니다.")

Detected source tables: ['FCT1_machine_log', 'FCT2_machine_log', 'FCT3_machine_log', 'FCT4_machine_log']


In [3]:
def load_union_logs(engine, schema: str, tables: list[str]) -> pd.DataFrame:
    sql_parts = []

    for t in tables:
        sql_parts.append(f"""
            SELECT
                end_day::text  AS end_day,
                station::text  AS station,
                end_time::text AS end_time,
                contents::text AS contents,
                '{t}'::text    AS src_table
            FROM "{schema}"."{t}"
            WHERE
                contents LIKE :ok || '%'
                OR contents LIKE :ng || '%'
                OR contents = :on_text
        """)

    union_sql = "\nUNION ALL\n".join(sql_parts)

    return pd.read_sql(
        text(union_sql),
        engine,
        params={
            "ok": TEST_OK_PREFIX,
            "ng": TEST_NG_PREFIX,
            "on_text": ON_TEXT,
        }
    )


df = load_union_logs(engine, SRC_SCHEMA, src_tables)
print(f"Loaded rows: {len(df):,}")
df.head(10)

Loaded rows: 651,449


,end_day,station,end_time,contents,src_table
0,20250808,FCT1,16:08:44.11,TEST RESULT :: OK,FCT1_machine_log
1,20250808,FCT1,16:08:51.83,제품 안착 완료 ON,FCT1_machine_log
2,20250808,FCT1,16:09:13.26,TEST RESULT :: OK,FCT1_machine_log
3,20250808,FCT1,16:09:21.44,제품 안착 완료 ON,FCT1_machine_log
4,20250808,FCT1,16:09:42.42,TEST RESULT :: OK,FCT1_machine_log
5,20250808,FCT1,16:09:51.30,제품 안착 완료 ON,FCT1_machine_log
6,20250808,FCT1,16:10:12.13,TEST RESULT :: OK,FCT1_machine_log
7,20250808,FCT1,16:10:20.47,제품 안착 완료 ON,FCT1_machine_log
8,20250808,FCT1,16:10:42.11,TEST RESULT :: OK,FCT1_machine_log
9,20250808,FCT1,16:10:50.09,제품 안착 완료 ON,FCT1_machine_log


In [4]:
# Cell 3) end_ts 생성 + op_gap 계산 (제품 안착 완료 ON 행에만 값, TEST RESULT 행은 null)

def build_end_ts(df: pd.DataFrame) -> pd.DataFrame:
    # end_day: 'YYYYMMDD' / end_time: 'HH:MM:SS' or 'HH:MM:SS.sss'
    # 안전하게 합쳐서 datetime으로 파싱
    s = df["end_day"].astype(str).str.strip() + " " + df["end_time"].astype(str).str.strip()
    df = df.copy()
    df["end_ts"] = pd.to_datetime(s, errors="coerce", format=None)
    bad = df["end_ts"].isna().sum()
    if bad:
        print(f"[WARN] end_ts 파싱 실패: {bad} rows (end_day/end_time 형식 확인 필요)")
    return df

def compute_op_gap(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # 정렬(요구사항: end_time 오름차순 / end_day, station 별 구분)
    df = df.sort_values(["end_day", "station", "end_ts"], ascending=True).reset_index(drop=True)

    # 그룹 내 직전 행 정보
    df["prev_end_ts"] = df.groupby(["end_day", "station"])["end_ts"].shift(1)
    df["prev_contents"] = df.groupby(["end_day", "station"])["contents"].shift(1)

    # 현재 행이 ON이고, 직전 행이 TEST RESULT(OK/NG)일 때만 gap 계산
    is_on = df["contents"].eq(ON_TEXT)
    prev_is_test = df["prev_contents"].fillna("").str.startswith(TEST_OK_PREFIX) | df["prev_contents"].fillna("").str.startswith(TEST_NG_PREFIX)

    gap_sec = (df["end_ts"] - df["prev_end_ts"]).dt.total_seconds()

    df["op_gap"] = np.where(is_on & prev_is_test, gap_sec, np.nan)

    # 소수점 3자리에서 반올림하여 2자리(=일반 round(2))
    df["op_gap"] = df["op_gap"].round(2)

    # 출력/저장 컬럼 구성 (TEST RESULT 행은 op_gap null 유지)
    out = df[["end_day", "station", "end_time", "contents", "op_gap", "src_table"]].copy()
    out = out.sort_values(["end_day", "end_time"], ascending=True).reset_index(drop=True)
    return out

df2 = build_end_ts(df)
df_out = compute_op_gap(df2)

df_out.head(20)

,end_day,station,end_time,contents,op_gap,src_table
0,20250808,FCT3,13:16:30.50,제품 안착 완료 ON,NaN,FCT3_machine_log
1,20250808,FCT3,13:16:51.41,TEST RESULT :: OK,NaN,FCT3_machine_log
2,20250808,FCT2,13:16:53.12,제품 안착 완료 ON,NaN,FCT2_machine_log
3,20250808,FCT1,13:16:55.72,TEST RESULT :: OK,NaN,FCT1_machine_log
4,20250808,FCT3,13:17:03.06,제품 안착 완료 ON,11.65,FCT3_machine_log
5,20250808,FCT1,13:17:09.49,제품 안착 완료 ON,13.77,FCT1_machine_log
6,20250808,FCT2,13:17:13.65,TEST RESULT :: OK,NaN,FCT2_machine_log
7,20250808,FCT2,13:17:21.87,제품 안착 완료 ON,8.22,FCT2_machine_log
8,20250808,FCT1,13:17:30.75,TEST RESULT :: OK,NaN,FCT1_machine_log
9,20250808,FCT3,13:17:31.69,TEST RESULT :: OK,NaN,FCT3_machine_log


In [5]:
# Cell 4) 결과 테이블 생성(없으면) + 저장
# - 요구사항: 스키마 d1_machine_log, 테이블 fct_op_gap (없으면 생성)
# - 운영 방식은 "매번 최신으로 교체" (replace)로 구현
#   필요하면 if_exists='append'로 변경 가능합니다.

create_schema_sql = f"CREATE SCHEMA IF NOT EXISTS {TARGET_SCHEMA};"

# to_sql이 테이블을 만들어주긴 하지만, 타입을 명확히 주고 싶으면 DDL을 먼저 만들 수 있습니다.
# 여기서는 간단히 replace로 처리합니다.
with engine.begin() as conn:
    conn.execute(text(create_schema_sql))

df_out.to_sql(
    name=TARGET_TABLE,
    con=engine,
    schema=TARGET_SCHEMA,
    if_exists="replace",   # 필요 시 "append"로 변경
    index=False
)

print(f"[OK] Saved {len(df_out):,} rows -> {TARGET_SCHEMA}.{TARGET_TABLE}")

[OK] Saved 651,449 rows -> d1_machine_log.fct_op_gap


In [6]:
# Cell 5) 최종 출력 (첨부 예시처럼)
# end_day, end_time 오름차순 기준 출력
df_out

,end_day,station,end_time,contents,op_gap,src_table
0,20250808,FCT3,13:16:30.50,제품 안착 완료 ON,NaN,FCT3_machine_log
1,20250808,FCT3,13:16:51.41,TEST RESULT :: OK,NaN,FCT3_machine_log
2,20250808,FCT2,13:16:53.12,제품 안착 완료 ON,NaN,FCT2_machine_log
3,20250808,FCT1,13:16:55.72,TEST RESULT :: OK,NaN,FCT1_machine_log
4,20250808,FCT3,13:17:03.06,제품 안착 완료 ON,11.65,FCT3_machine_log
...,...,...,...,...,...,...
651444,20251120,FCT4,13:49:25.82,TEST RESULT :: OK,NaN,FCT4_machine_log
651445,20251120,FCT3,13:49:30.66,제품 안착 완료 ON,8.28,FCT3_machine_log
651446,20251120,FCT2,13:49:38.76,TEST RESULT :: NG,NaN,FCT2_machine_log
651447,20251120,FCT1,13:49:40.52,제품 안착 완료 ON,22.28,FCT1_machine_log


In [7]:
# Cell 6) station별 op_gap Boxplot 요약 + fct_op_gap_av 저장

from sqlalchemy import text
import numpy as np
import pandas as pd

SUMMARY_SCHEMA = "d1_machine_log"
SUMMARY_TABLE  = "fct_op_gap_av"

def fmt_range(a, b, decimals=1):
    # 예: 16.0~22.0 형태
    if a is None or b is None or (isinstance(a, float) and np.isnan(a)) or (isinstance(b, float) and np.isnan(b)):
        return None
    return f"{a:.{decimals}f}~{b:.{decimals}f}"

def compute_boxplot_summary(df: pd.DataFrame) -> pd.DataFrame:
    # op_gap이 있는 행만 대상
    d = df.loc[df["op_gap"].notna(), ["station", "op_gap"]].copy()
    d["op_gap"] = pd.to_numeric(d["op_gap"], errors="coerce")
    d = d.dropna(subset=["op_gap"])

    rows = []
    stations = sorted(d["station"].unique())  # FCT1~4

    for st in stations:
        s = d.loc[d["station"] == st, "op_gap"].astype(float)

        if len(s) == 0:
            continue

        q1 = float(s.quantile(0.25))
        med = float(s.quantile(0.50))
        q3 = float(s.quantile(0.75))
        iqr = q3 - q1

        # Tukey fences (boxplot 기준)
        lower_fence = q1 - 1.5 * iqr
        upper_fence = q3 + 1.5 * iqr

        # 실제 데이터 중 fence 밖은 outlier
        lower_outliers = s[s < lower_fence]
        upper_outliers = s[s > upper_fence]

        # "lower 이상치 범위": (lower_outliers의 min ~ max)  (없으면 None)
        # "upper 이상치 범위": (upper_outliers의 min ~ max)  (없으면 None)
        lo_min = float(lower_outliers.min()) if len(lower_outliers) else np.nan
        lo_max = float(lower_outliers.max()) if len(lower_outliers) else np.nan
        up_min = float(upper_outliers.min()) if len(upper_outliers) else np.nan
        up_max = float(upper_outliers.max()) if len(upper_outliers) else np.nan

        # 이상치 제외 (fence 안쪽만)
        inlier = s[(s >= lower_fence) & (s <= upper_fence)]
        del_out_mean = float(inlier.mean()) if len(inlier) else np.nan

        rows.append({
            "station": st,
            "op_gap_lower_outlier": fmt_range(lo_min, lo_max, decimals=1),
            "q1": round(q1, 1),
            "median": round(med, 1),
            "q3": round(q3, 1),
            "op_gap_upper_outlier": fmt_range(up_min, up_max, decimals=1),
            "del_out_op_gap_av": round(del_out_mean, 2),
            "lower_fence": round(lower_fence, 3),  # 디버그/검증용(원하면 삭제 가능)
            "upper_fence": round(upper_fence, 3),  # 디버그/검증용(원하면 삭제 가능)
            "n": int(len(s)),
            "n_inlier": int(len(inlier)),
        })

    out = pd.DataFrame(rows)

    # 첨부 예시처럼 id 부여
    out = out.sort_values("station").reset_index(drop=True)
    out.insert(0, "id", np.arange(1, len(out) + 1))

    # 최종 출력 컬럼(예시 표에 맞춤)
    # lower_fence/upper_fence/n/n_inlier는 원하면 숨겨도 됨
    out_view = out[[
        "id",
        "station",
        "op_gap_lower_outlier",
        "q1",
        "median",
        "q3",
        "op_gap_upper_outlier",
        "del_out_op_gap_av",
        "lower_fence",
        "upper_fence",
        "n",
        "n_inlier",
    ]].copy()

    return out_view

df_op_gap_av = compute_boxplot_summary(df_out)

# 저장 (없으면 생성)
with engine.begin() as conn:
    conn.execute(text(f'CREATE SCHEMA IF NOT EXISTS "{SUMMARY_SCHEMA}";'))

df_op_gap_av.to_sql(
    name=SUMMARY_TABLE,
    con=engine,
    schema=SUMMARY_SCHEMA,
    if_exists="replace",
    index=False
)

print(f"[OK] Saved {len(df_op_gap_av):,} rows -> {SUMMARY_SCHEMA}.{SUMMARY_TABLE}")

df_op_gap_av

[OK] Saved 4 rows -> d1_machine_log.fct_op_gap_av


,id,station,op_gap_lower_outlier,q1,median,q3,op_gap_upper_outlier,del_out_op_gap_av,lower_fence,upper_fence,n,n_inlier
0,1,FCT1,5.1~5.1,8.0,8.3,9.4,11.4~50964.9,8.49,6.030,11.390,81516,69809
1,2,FCT2,6.1~7.6,8.4,8.6,9.0,9.9~49094.6,8.64,7.595,9.875,79196,66974
2,3,FCT3,5.6~7.3,8.2,8.4,8.7,9.6~51542.0,8.37,7.345,9.545,82976,73196
3,4,FCT4,None,7.7,8.1,10.5,14.7~51559.1,8.89,3.500,14.700,80991,75215


In [8]:
# Cell (FINAL-FIX) 마지막 dataframe만 원격 DB에 저장 (Tailscale/VPN 안정 설정)

import urllib.parse
import time
from sqlalchemy import create_engine, text
from sqlalchemy.exc import OperationalError

DB_CONFIG = {
    "host": "100.105.75.47",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "",
}

TARGET_SCHEMA = "d1_machine_log"
TARGET_TABLE  = "fct_op_gap_av"

def get_engine_remote(cfg):
    user = cfg["user"]
    password = urllib.parse.quote_plus(cfg["password"])
    host = cfg["host"]
    port = cfg["port"]
    dbname = cfg["dbname"]

    # ✅ 핵심: sslmode=disable (DBeaver와 동일)
    # ✅ connect_timeout은 20~30 권장 (DBeaver도 수십초 걸릴 수 있음)
    conn_str = (
        f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}"
        "?connect_timeout=30&sslmode=disable"
    )

    # ✅ 연결 흔들림 대비: pre_ping / recycle / 풀 사이즈 최소화
    return create_engine(
        conn_str,
        pool_pre_ping=True,
        pool_recycle=30,
        pool_size=1,
        max_overflow=0,
    )

def connect_test(engine, retries=6, sleep_sec=5):
    last = None
    for i in range(1, retries + 1):
        try:
            with engine.connect() as conn:
                conn.execute(text("SELECT 1"))
            print(f"[OK] Connection test success ({i}/{retries})")
            return
        except OperationalError as e:
            last = e
            print(f"[WARN] Connection test failed ({i}/{retries}) -> {e}")
            time.sleep(sleep_sec)
    raise last

engine_remote = get_engine_remote(DB_CONFIG)

# 1) 연결 테스트(재시도 포함)
connect_test(engine_remote, retries=6, sleep_sec=5)

# 2) 스키마 보장 + 저장
with engine_remote.begin() as conn:
    conn.execute(text(f'CREATE SCHEMA IF NOT EXISTS "{TARGET_SCHEMA}";'))

df_op_gap_av.to_sql(
    name=TARGET_TABLE,
    con=engine_remote,
    schema=TARGET_SCHEMA,
    if_exists="replace",
    index=False,
    method="multi",
    chunksize=2000,
)

print(f"[OK] Saved final dataframe -> {TARGET_SCHEMA}.{TARGET_TABLE} @ {DB_CONFIG['host']}")

[OK] Connection test success (1/6)
[OK] Saved final dataframe -> d1_machine_log.fct_op_gap_av @ 100.105.75.47
